# CS224N · Lecture 7 — Attention, MT Evaluation, Final Projects, and Practical Tips

**Slides:** [cs224n-spr2024-lecture07-final-project.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture07-final-project.pdf)  
**Prerequisite:** the Lecture 6 notebook (LSTM encoder–decoder). This notebook reuses the same toy translation task, and loads Lecture 6's trained model if it finds `seq2seq_no_attention.npz` in the working directory (otherwise it retrains it).

### What you will build
| § | Topic | You will implement |
|---|---|---|
| 1 | **Evaluating MT: BLEU** | BLEU from scratch, verified against hand computation; its failure modes |
| 2 | 🔎 Decoding: greedy vs. **beam search** | Beam search with length normalization |
| 3 | **Attention**: the bottleneck problem and its solution | Seq2seq + attention from scratch (gradient-checked); alignment heatmaps |
| 4 | Attention variants and attention as a general technique | Dot, multiplicative, reduced-rank, additive scoring |
| 5 | Final projects: types, topics, data | — |
| 6 | **Care with datasets**: train / tune / dev / test | Overfitting to the dev set, demonstrated |
| 7 | **Getting your neural network to train** | Tiny-batch sanity check; catching a data-leak bug; error analysis |
| 8 | PyTorch version of attention NMT | — |

> **Attention** "has become the powerful, flexible, general way pointer and memory manipulation in all deep learning models. A new idea from after 2010! From NMT!" It's also the core of the Transformer (next lecture).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, time, collections, os

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

class Adam:
    def __init__(self, params, lr=2e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
    def step(self, params, grads):
        self.t += 1
        for k in params:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            params[k] -= self.lr * (self.m[k] / (1 - self.b1 ** self.t)) / (np.sqrt(self.v[k] / (1 - self.b2 ** self.t)) + self.eps)

def clip_grads(g, max_norm=5.0):
    norm = math.sqrt(sum((v ** 2).sum() for v in g.values()))
    if norm > max_norm:
        for k in g:
            g[k] *= max_norm / norm
    return norm

The toy translation task and LSTM layer from Lecture 6 (same generator and seed, so the data are identical). Source: English-like SVO. Target "Toy": verb-final, adjectives after the noun in mirrored order, postpositions.

In [ ]:
LEX = {"the": "le", "a": "un", "dog": "kaz", "cat": "miro", "bird": "tupi", "farmer": "olan", "child": "bebi",
       "teacher": "sanu", "ball": "pilo", "book": "libu", "apple": "mazo", "letter": "kato",
       "big": "gros", "small": "pik", "red": "roj", "old": "vej", "happy": "felo",
       "sees": "vidas", "eats": "manjas", "finds": "trovas", "throws": "jetas", "likes": "amas",
       "in": "ene", "near": "apud", "with": "kun", "garden": "jardo", "house": "domo", "park": "parko", ".": "."}
N_ANIM = ["dog", "cat", "bird", "farmer", "child", "teacher"]
N_OBJ = ["ball", "book", "apple", "letter"] + N_ANIM
N_PLACE = ["garden", "house", "park"]
ADJS = ["big", "small", "red", "old", "happy"]
VERBS_MT = ["sees", "eats", "finds", "throws", "likes"]
PREPS = ["in", "near", "with"]

def noun_phrase(nouns, max_adj, rng):
    det = rng.choice(["the", "a"]); adjs = [rng.choice(ADJS) for _ in range(rng.choice(range(max_adj + 1)))]
    n = rng.choice(nouns)
    return [det] + adjs + [n], [LEX[det], LEX[n]] + [LEX[a] for a in reversed(adjs)]

def translation_pair(rng=random, max_adj=2, max_pp=2):
    s_subj, t_subj = noun_phrase(N_ANIM, max_adj, rng)
    verb = rng.choice(VERBS_MT)
    s_obj, t_obj = noun_phrase(N_OBJ, max_adj, rng)
    src, tgt = s_subj + [verb] + s_obj, t_subj + t_obj
    for _ in range(rng.choice(range(max_pp + 1))):
        prep = rng.choice(PREPS); s_np, t_np = noun_phrase(N_PLACE + N_ANIM, max_adj, rng)
        src += [prep] + s_np; tgt += t_np + [LEX[prep]]
    return src + ["."], tgt + [LEX[verb], "."]

random.seed(0)
mt_data = [translation_pair() for _ in range(10000)]
mt_train, mt_test = mt_data[:8500], mt_data[8500:]
src_vocab = ["<pad>"] + sorted({w for s, _ in mt_data for w in s})
tgt_vocab = ["<pad>", "<s>"] + sorted({w for _, t in mt_data for w in t})
S2I = {w: i for i, w in enumerate(src_vocab)}
T2I = {w: i for i, w in enumerate(tgt_vocab)}

def mt_batch(pairs):
    B = len(pairs); Ls = max(len(s) for s, _ in pairs); Lt = max(len(t) for _, t in pairs)
    S = np.zeros((B, Ls), int); SM = np.zeros((B, Ls))
    Tin = np.zeros((B, Lt), int); Tout = np.zeros((B, Lt), int); TM = np.zeros((B, Lt))
    for b, (s, t) in enumerate(pairs):
        S[b, :len(s)] = [S2I[w] for w in s]; SM[b, :len(s)] = 1
        ids = [T2I[w] for w in t]
        Tin[b, :len(t)] = [T2I["<s>"]] + ids[:-1]; Tout[b, :len(t)] = ids; TM[b, :len(t)] = 1
    return S, SM, Tin, Tout, TM

def lstm_init(d, H, r, forget_bias=1.0):
    W = r.normal(0, 1 / np.sqrt(d + H), (4 * H, d + H)); b = np.zeros(4 * H); b[:H] = forget_bias
    return W, b

def lstm_forward(W, b, X, mask, h0, c0):
    B, T, d = X.shape; H = h0.shape[1]; h, c = h0, c0; Hs = np.zeros((B, T, H)); cache = []
    for t in range(T):
        xh = np.concatenate([X[:, t], h], 1); a = xh @ W.T + b
        f, i, o = sigmoid(a[:, :H]), sigmoid(a[:, H:2 * H]), sigmoid(a[:, 2 * H:3 * H]); g = np.tanh(a[:, 3 * H:])
        cn = f * c + i * g; tc = np.tanh(cn); hn = o * tc; m = mask[:, t:t + 1]
        cache.append((xh, f, i, o, g, c, tc, m)); h = m * hn + (1 - m) * h; c = m * cn + (1 - m) * c; Hs[:, t] = h
    return Hs, h, c, cache

def lstm_backward(W, dHs, dhT, dcT, cache, d):
    B, T, H = dHs.shape; dX = np.zeros((B, T, d)); gW = np.zeros_like(W); gb = np.zeros(W.shape[0])
    dh, dc = dhT.copy(), dcT.copy()
    for t in reversed(range(T)):
        xh, f, i, o, g, c_prev, tc, m = cache[t]; dh = dh + dHs[:, t]
        dhn, dcin = m * dh, m * dc; dh_prev, dc_prev = (1 - m) * dh, (1 - m) * dc
        do = dhn * tc; dct = dhn * o * (1 - tc ** 2) + dcin
        df, di, dg = dct * c_prev, dct * g, dct * i; dc_prev = dc_prev + dct * f
        da = np.concatenate([df * f * (1 - f), di * i * (1 - i), do * o * (1 - o), dg * (1 - g ** 2)], 1)
        gW += da.T @ xh; gb += da.sum(0); dxh = da @ W; dX[:, t] = dxh[:, :d]; dh_prev = dh_prev + dxh[:, d:]
        dh, dc = dh_prev, dc_prev
    return dX, dh, dc, gW, gb

---
## 1. How do we evaluate machine translation?

**Commonest way: BLEU (Bilingual Evaluation Understudy)**, Papineni et al. 2002. BLEU compares the machine-written translation to **one or several human-written translations**, and computes a similarity score based on:
* the **geometric mean of n-gram precision** (usually for 1, 2, 3 and 4-grams)
* plus a **penalty for too-short system translations**

### 1.1 The formula
**Modified (clipped) n-gram precision.** For each n-gram in the candidate, count it at most as many times as it appears in the reference that has the most of it:
$$p_n = \frac{\sum_{\text{n-gram}\in\text{cand}}\min\big(\text{count}_{\text{cand}}(\text{n-gram}),\ \max_{\text{ref}}\text{count}_{\text{ref}}(\text{n-gram})\big)}{\sum_{\text{n-gram}\in\text{cand}}\text{count}_{\text{cand}}(\text{n-gram})}$$
(Without clipping, "the the the the the the the" would get unigram precision 7/7 against "the cat is on the mat".)

**Brevity penalty.** Precision alone rewards outputting only the few words you're sure of. With candidate length $c$ and (closest) reference length $r$:
$$\text{BP} = \begin{cases}1 & c > r\\ e^{\,1 - r/c} & c\leq r\end{cases}$$

$$\text{BLEU} = \text{BP}\cdot\exp\Big(\sum_{n=1}^{4}\tfrac14\log p_n\Big)$$

At **corpus level**, the clipped counts and totals are summed over all sentences *before* dividing (not averaged per sentence).

In [ ]:
def ngram_counts(tokens, n):
    return collections.Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def corpus_bleu(candidates, references_list, max_n=4, smooth=False):
    # candidates: list of token lists; references_list: list of lists of reference token lists.
    clipped, total = [0] * max_n, [0] * max_n
    c_len = r_len = 0
    for cand, refs in zip(candidates, references_list):
        c_len += len(cand)
        r_len += min(refs, key=lambda ref: (abs(len(ref) - len(cand)), len(ref))).__len__()   # closest ref length
        for n in range(1, max_n + 1):
            cand_counts = ngram_counts(cand, n)
            max_ref = collections.Counter()
            for ref in refs:
                for g, k in ngram_counts(ref, n).items():
                    max_ref[g] = max(max_ref[g], k)
            clipped[n - 1] += sum(min(k, max_ref[g]) for g, k in cand_counts.items())
            total[n - 1] += max(len(cand) - n + 1, 0)
    if smooth:      # add-one smoothing for n > 1 (Chen & Cherry 2014, method 2): for sentence-level BLEU
        precisions = [clipped[0] / total[0]] + [(clipped[n] + 1) / (total[n] + 1) for n in range(1, max_n)]
    else:
        if min(clipped) == 0:
            return 0.0
        precisions = [c / t for c, t in zip(clipped, total)]
    bp = 1.0 if c_len > r_len else math.exp(1 - r_len / c_len)
    return bp * math.exp(sum(math.log(p) for p in precisions) / max_n)

ref = "the cat is on the mat".split()
print("identical               :", corpus_bleu([ref], [[ref]]))
print("'the the the ...' BLEU-1:", round(corpus_bleu(["the the the the the the the".split()], [[ref]], max_n=1), 4),
      "(clipped unigram precision 2/7, Papineni et al.'s example)")
cand = "the cat sat on the mat".split()
print("'the cat sat on the mat' BLEU-3:", round(corpus_bleu([cand], [[ref]], max_n=3), 4),
      " by hand: (5/6 * 3/5 * 1/4)^(1/3) =", round((5 / 6 * 3 / 5 * 1 / 4) ** (1 / 3), 4))
print("...its BLEU-4:", corpus_bleu([cand], [[ref]]), "(no 4-gram matches -> 0 without smoothing)")
print("short candidate 'the cat':", round(corpus_bleu(["the cat".split()], [[ref]], max_n=2), 4), "(brevity penalty at work)")

### 1.2 BLEU is useful but imperfect
* **There are many valid ways to translate a sentence.** So a good translation can get a poor BLEU score because it has low n-gram overlap with the human translation. ☹
* The original BLEU paper used **4 reference translations** (the slides' example: a garbled machine translation about Guam airport receiving a threatening e-mail, scored against four quite different human translations). But **commonly now there is only one reference**, so the results are more "in expectation".
* It only measures **surface overlap**: it can't tell a harmless synonym from a meaning-reversing error.

In [ ]:
reference = "the committee approved the new budget on monday".split()
candidates = {
    "valid paraphrase": "on monday the panel passed the new budget",
    "one word changed, meaning reversed": "the committee rejected the new budget on monday",
    "fluent but wrong content": "the committee approved the new library on sunday",
}
for name, c in candidates.items():
    print(f"{name:36s} BLEU = {corpus_bleu([c.split()], [[reference]], smooth=True):.3f}   '{c}'")

The paraphrase, a perfect translation, scores far *below* the translation that flips the meaning. That's why BLEU is best used to compare systems over a whole **corpus**, where these quirks average out somewhat, and never as the only evidence.

🔎 **In practice today:**
* Always report BLEU with **sacreBLEU** (Post 2018), which standardizes tokenization. BLEU numbers computed with different tokenizers are not comparable.
* **chrF** (character n-gram F-score) is more robust for morphologically rich languages.
* **Learned metrics** like **COMET** and **BLEURT** (neural models trained to predict human judgments) correlate much better with humans, and are now standard at WMT alongside human evaluation.

**MT progress over time** (the slides' chart, Edinburgh En–De WMT newstest, cased BLEU): phrase-based SMT and syntax-based SMT plateaued around 20–25 BLEU in 2013–2015; **neural MT** jumped past them in 2015–2016 and kept climbing to over 40 by 2019.

---
## 2. 🔎 Decoding: greedy decoding vs. beam search

*(Not in these slides, but you need it for Assignment 3 and for every real seq2seq system.)*

The decoder gives $P(y_t\mid y_{<t}, x)$. We want $\arg\max_y P(y\mid x) = \arg\max_y\prod_t P(y_t\mid y_{<t}, x)$, but there are $|V|^T$ possible outputs.

* **Greedy decoding** (Lecture 6) takes the argmax at each step. **Problem: no way to undo decisions.** A locally best word can lead to a globally bad sentence.
* **Exhaustive search** is $O(|V|^T)$: far too expensive.
* **Beam search:** on each step, keep track of the **$k$ most probable partial translations** (hypotheses); $k$ is the **beam size** (in practice around 5 to 10). A hypothesis $y_1,\dots,y_t$ has score $\sum_{i=1}^{t}\log P(y_i\mid y_{<i}, x)$ (all negative; higher is better).
    * Expand each of the $k$ hypotheses with its top candidates, keep the best $k$ overall, repeat.
    * When a hypothesis produces the end token, it's **complete**: set it aside and continue with the others.
    * Stop at a max length $T$, or once we have $n$ completed hypotheses.
    * Not guaranteed to find the optimal solution, but much more efficient than exhaustive search.
* **Length normalization:** longer hypotheses have lower (more negative) scores, so beam search prefers short outputs. Fix: choose the final hypothesis by $\frac{1}{t}\sum_{i=1}^{t}\log P(y_i\mid y_{<i}, x)$.

We'll implement beam search once we have a model with attention (§3), and compare it to greedy decoding on both models.

---
## 3. Why attention? The sequence-to-sequence bottleneck problem

In the seq2seq model, the **encoding of the source sentence** (the encoder's final state) needs to **capture all information about the source sentence**. **Information bottleneck!**

**Attention** provides a solution to the bottleneck problem.
> **Core idea:** on each step of the decoder, use a **direct connection to the encoder** to **focus on a particular part** of the source sequence.

### 3.1 Seq2seq with attention, step by step (the slides' diagram)
On decoder step 1 (input `<START>`, source *il a m' entarté*):
1. Compute **attention scores**: the dot product of the decoder hidden state with **each** encoder hidden state.
2. Take **softmax** to turn the scores into a probability distribution, the **attention distribution**. (On this step, the model mostly focuses on the first encoder hidden state, *"il"* → *he*.)
3. Use the attention distribution to take a **weighted sum of the encoder hidden states**: the **attention output**. It mostly contains information from the hidden states that received high attention.
4. **Concatenate** the attention output with the decoder hidden state, then use it to compute $\hat y_1$ as before → *he*.
5. Repeat on every decoder step: *hit*, *me*, *with*, *a*, *pie*.

Sometimes we also take the attention output from the previous step and feed it into the decoder along with the usual decoder input ("**input feeding**"; used in Assignment 4).

### 3.2 Attention in equations
* Encoder hidden states $h_1, \dots, h_N\in\mathbb{R}^{h}$.
* On timestep $t$, decoder hidden state $s_t\in\mathbb{R}^h$.
* **Attention scores** for this step: $e^t = [\,s_t^\top h_1, \dots, s_t^\top h_N\,]\in\mathbb{R}^N$
* **Attention distribution** (a probability distribution, sums to 1): $\alpha^t = \text{softmax}(e^t)\in\mathbb{R}^N$
* **Attention output** (weighted sum of encoder states): $a_t = \sum_{i=1}^{N}\alpha_i^t h_i\in\mathbb{R}^h$
* Finally, **concatenate** the attention output with the decoder hidden state, $[a_t; s_t]\in\mathbb{R}^{2h}$, and proceed as in the non-attention model.

**Our implementation** uses the **multiplicative** score $e_i = s_t^\top W h_i$ (§4; Luong, Pham & Manning 2015, the variant used in the assignments), and combines with $o_t = \tanh(W_c[a_t; s_t])$, $\hat y_t = \text{softmax}(U o_t)$.

### 3.3 Backprop through attention
Everything is differentiable. Working backwards from $\delta_o = \partial J/\partial o_t$:
* $\delta_z = \delta_o\odot(1 - o_t^2)$, giving $\partial J/\partial W_c$, and splitting $W_c^\top\delta_z$ into $\delta_a$ (for $a_t$) and part of $\delta_s$
* $a_t = \sum_i\alpha_i h_i$: $\ \partial J/\partial\alpha_i = \delta_a^\top h_i$, and **every encoder state gets a direct gradient** $\alpha_i\,\delta_a$
* softmax: $\ \partial J/\partial e_i = \alpha_i\big(\partial J/\partial\alpha_i - \sum_j\alpha_j\,\partial J/\partial\alpha_j\big)$
* $e_i = s_t^\top W h_i$: gradients to $s_t$, to $W$, and **another direct gradient** to each $h_i$

That "direct gradient to every encoder state" is the point: **attention gives gradients a shortcut to faraway states** (no long chain of Jacobians).

In [ ]:
def init_attn_seq2seq(d=32, H=64, seed=0):
    r = np.random.default_rng(seed)
    p = {"Es": r.normal(0, 0.1, (len(src_vocab), d)), "Et": r.normal(0, 0.1, (len(tgt_vocab), d)),
         "Wa": r.normal(0, 1 / np.sqrt(H), (H, H)),              # multiplicative attention: e_i = s^T Wa h_i
         "Wc": r.normal(0, 1 / np.sqrt(2 * H), (H, 2 * H)),       # combine [a; s]
         "U": r.normal(0, 1 / np.sqrt(H), (len(tgt_vocab), H)), "c": np.zeros(len(tgt_vocab))}
    p["encW"], p["encb"] = lstm_init(d, H, r)
    p["decW"], p["decb"] = lstm_init(d, H, r)
    return p

def attn_seq2seq_fb(p, S, SM, Tin, Tout, TM, need_grad=True):
    B = S.shape[0]; H = p["U"].shape[1]; d = p["Es"].shape[1]; z = np.zeros((B, H))
    Henc, hT, cT, enc_cache = lstm_forward(p["encW"], p["encb"], p["Es"][S], SM, z, z)         # (B, N, H)
    Sdec, _, _, dec_cache = lstm_forward(p["decW"], p["decb"], p["Et"][Tin], np.ones(Tin.shape), hT, cT)  # (B, T, H)
    K = Henc @ p["Wa"].T                                             # (B, N, H): Wa h_i for every source position
    e = np.einsum("bth,bnh->btn", Sdec, K)                           # scores e[t, i] = s_t . Wa h_i
    e = np.where(SM[:, None, :] > 0, e, -1e9)                        # never attend to padding
    alpha = softmax(e, -1)                                           # attention distributions (B, T, N)
    A = alpha @ Henc                                                 # attention outputs (B, T, H)
    AS = np.concatenate([A, Sdec], -1)
    O = np.tanh(AS @ p["Wc"].T)
    P = softmax(O @ p["U"].T + p["c"])
    Bi, Ti = np.meshgrid(np.arange(B), np.arange(Tout.shape[1]), indexing="ij")
    ntok = TM.sum()
    loss = -(np.log(P[Bi, Ti, Tout] + 1e-12) * TM).sum() / ntok
    if not need_grad:
        return loss, P, alpha
    G = P.copy(); G[Bi, Ti, Tout] -= 1; G *= TM[:, :, None] / ntok
    g = {"U": np.einsum("btv,bth->vh", G, O), "c": G.sum((0, 1))}
    dZ = (G @ p["U"]) * (1 - O ** 2)
    g["Wc"] = np.einsum("bth,btk->hk", dZ, AS)
    dAS = dZ @ p["Wc"]; dA, dS = dAS[..., :H], dAS[..., H:].copy()
    dalpha = dA @ Henc.transpose(0, 2, 1)
    dHenc = alpha.transpose(0, 2, 1) @ dA                            # direct gradient into every encoder state
    de = alpha * (dalpha - (dalpha * alpha).sum(-1, keepdims=True))  # softmax backward
    dS += de @ K
    dK = de.transpose(0, 2, 1) @ Sdec
    g["Wa"] = np.einsum("bnh,bnk->hk", dK, Henc)
    dHenc += dK @ p["Wa"]                                            # ...and a second direct path through the scores
    dXt, dh0, dc0, g["decW"], g["decb"] = lstm_backward(p["decW"], dS, z, z, dec_cache, d)
    dXs, _, _, g["encW"], g["encb"] = lstm_backward(p["encW"], dHenc, dh0, dc0, enc_cache, d)
    g["Et"] = np.zeros_like(p["Et"]); np.add.at(g["Et"], Tin, dXt)
    g["Es"] = np.zeros_like(p["Es"]); np.add.at(g["Es"], S, dXs)
    return loss, g

# Full gradient check on a tiny model
p_chk = init_attn_seq2seq(d=4, H=3, seed=1)
bt = mt_batch(mt_train[:3])
_, g_chk = attn_seq2seq_fb(p_chk, *bt)
used_src, used_tgt = set(bt[0].ravel()), set(bt[2].ravel())
print(f"{'param':6s} {'max abs error':>14s} {'max |grad|':>11s}")
for k in p_chk:
    num = np.zeros_like(p_chk[k])
    for idx in np.ndindex(p_chk[k].shape):
        if (k == "Es" and idx[0] not in used_src) or (k == "Et" and idx[0] not in used_tgt):
            continue
        old = p_chk[k][idx]
        p_chk[k][idx] = old + 1e-6; lp = attn_seq2seq_fb(p_chk, *bt, need_grad=False)[0]
        p_chk[k][idx] = old - 1e-6; lm_ = attn_seq2seq_fb(p_chk, *bt, need_grad=False)[0]
        p_chk[k][idx] = old
        num[idx] = (lp - lm_) / 2e-6
    print(f"{k:6s} {np.abs(num - g_chk[k]).max():14.1e} {np.abs(g_chk[k]).max():11.1e}")

**A gradient-checking lesson.** All absolute errors are ~$10^{-10}$, the finite-difference noise floor, so the gradients are correct. But notice `Wa`'s gradient is tiny at initialization: random attention is nearly uniform, so changing the scores barely matters yet. A *relative*-error check would flag `Wa` as suspicious (noise / tiny gradient = large ratio), a classic false alarm. **When a gradient is tiny, compare absolute errors with the noise floor**, or perturb the parameters away from the symmetric initialization before checking.

### 3.4 Training both models
We load (or retrain) Lecture 6's no-attention model, and train the attention model with the same data, sizes, and optimizer.

In [ ]:
def init_seq2seq(d=32, H=64, seed=0):
    r = np.random.default_rng(seed)
    p = {"Es": r.normal(0, 0.1, (len(src_vocab), d)), "Et": r.normal(0, 0.1, (len(tgt_vocab), d)),
         "U": r.normal(0, 1 / np.sqrt(H), (len(tgt_vocab), H)), "c": np.zeros(len(tgt_vocab))}
    p["encW"], p["encb"] = lstm_init(d, H, r); p["decW"], p["decb"] = lstm_init(d, H, r)
    return p

def seq2seq_fb(p, S, SM, Tin, Tout, TM, need_grad=True):
    B = S.shape[0]; H = p["U"].shape[1]; d = p["Es"].shape[1]; z = np.zeros((B, H))
    _, hT, cT, enc_cache = lstm_forward(p["encW"], p["encb"], p["Es"][S], SM, z, z)
    Hd, _, _, dec_cache = lstm_forward(p["decW"], p["decb"], p["Et"][Tin], np.ones(Tin.shape), hT, cT)
    P = softmax(Hd @ p["U"].T + p["c"])
    Bi, Ti = np.meshgrid(np.arange(B), np.arange(Tout.shape[1]), indexing="ij"); ntok = TM.sum()
    loss = -(np.log(P[Bi, Ti, Tout] + 1e-12) * TM).sum() / ntok
    if not need_grad:
        return loss, P
    G = P.copy(); G[Bi, Ti, Tout] -= 1; G *= TM[:, :, None] / ntok
    g = {"U": np.einsum("btv,bth->vh", G, Hd), "c": G.sum((0, 1))}
    dXt, dh0, dc0, g["decW"], g["decb"] = lstm_backward(p["decW"], G @ p["U"], z, z, dec_cache, d)
    dXs, _, _, g["encW"], g["encb"] = lstm_backward(p["encW"], np.zeros((B, S.shape[1], H)), dh0, dc0, enc_cache, d)
    g["Et"] = np.zeros_like(p["Et"]); np.add.at(g["Et"], Tin, dXt)
    g["Es"] = np.zeros_like(p["Es"]); np.add.at(g["Es"], S, dXs)
    return loss, g

def train(fb, p, epochs, label):
    opt, r, t0 = Adam(p), np.random.default_rng(0), time.time()
    for epoch in range(epochs):
        order = r.permutation(len(mt_train)); total = 0.0
        for k in range(0, len(order), 64):
            loss, g = fb(p, *mt_batch([mt_train[i] for i in order[k:k + 64]]))
            clip_grads(g); opt.step(p, g); total += loss
        print(f"[{label}] epoch {epoch + 1}: train loss {total / math.ceil(len(order) / 64):.4f}  ({time.time() - t0:.0f}s)")
    return p

if os.path.exists("seq2seq_no_attention.npz"):
    p_plain = dict(np.load("seq2seq_no_attention.npz"))
    print("loaded Lecture 6's no-attention model")
else:
    p_plain = train(seq2seq_fb, init_seq2seq(), 10, "no attention")
p_attn = train(attn_seq2seq_fb, init_attn_seq2seq(), 8, "attention")

### 3.5 Decoding: greedy and beam search for both models
One decoder step, vectorized over the $k$ hypotheses in the beam:

In [ ]:
def encode(p, src):
    H = p["U"].shape[1]; S = np.array([[S2I[w] for w in src]])
    Henc, h, c, _ = lstm_forward(p["encW"], p["encb"], p["Es"][S], np.ones(S.shape), np.zeros((1, H)), np.zeros((1, H)))
    return Henc[0], h, c

def decoder_step(p, Henc, y_ids, h, c):
    # One step for k hypotheses: y_ids (k,), h, c (k, H). Returns log-probs (k, V), new h, c, attention (k, N) or None.
    _, h, c, _ = lstm_forward(p["decW"], p["decb"], p["Et"][y_ids][:, None, :], np.ones((len(y_ids), 1)), h, c)
    if "Wa" in p:
        alpha = softmax(h @ (Henc @ p["Wa"].T).T, -1)          # (k, N)
        o = np.tanh(np.concatenate([alpha @ Henc, h], 1) @ p["Wc"].T)
    else:
        alpha, o = None, h
    logits = o @ p["U"].T + p["c"]
    return logits - np.log(np.exp(logits - logits.max(1, keepdims=True)).sum(1, keepdims=True)) - logits.max(1, keepdims=True), h, c, alpha

def greedy(p, src, max_len=40):
    Henc, h, c = encode(p, src); y, out, atts = T2I["<s>"], [], []
    for _ in range(max_len):
        logp, h, c, a = decoder_step(p, Henc, np.array([y]), h, c)
        y = int(logp[0].argmax()); out.append(tgt_vocab[y]); atts.append(a)
        if out[-1] == ".":
            break
    return out, atts

def beam_search(p, src, k=5, max_len=40, length_norm=True):
    Henc, h, c = encode(p, src)
    hyps = [([], 0.0)]; H_ = np.repeat(h, 1, 0); C_ = np.repeat(c, 1, 0); last = np.array([T2I["<s>"]])
    completed = []
    for _ in range(max_len):
        logp, H_, C_, _ = decoder_step(p, Henc, last, H_, C_)
        cand = []                                        # (score, hyp index, word)
        for hi, (toks, score) in enumerate(hyps):
            for w in np.argsort(-logp[hi])[:k]:
                cand.append((score + logp[hi, w], hi, int(w)))
        cand.sort(key=lambda x: -x[0])
        new_hyps, keep_idx, new_last = [], [], []
        for score, hi, w in cand:
            toks = hyps[hi][0] + [tgt_vocab[w]]
            if tgt_vocab[w] == ".":
                completed.append((toks, score))          # complete: set aside
            else:
                new_hyps.append((toks, score)); keep_idx.append(hi); new_last.append(w)
            if len(new_hyps) == k:
                break
        if len(completed) >= k or not new_hyps:
            break
        hyps, H_, C_, last = new_hyps, H_[keep_idx], C_[keep_idx], np.array(new_last)
    completed = completed or hyps
    key = (lambda x: x[1] / len(x[0])) if length_norm else (lambda x: x[1])
    return max(completed, key=key)[0]

def length_bucket(n):
    return "short (<=10)" if n <= 10 else ("medium (11-15)" if n <= 15 else "long (16+)")

def evaluate(decode, pairs):
    hyps, refs = [decode(s) for s, _ in pairs], [[t] for _, t in pairs]
    by_len = collections.defaultdict(list)
    for (s, t), hyp in zip(pairs, hyps):
        by_len[length_bucket(len(s))].append((hyp, t))
    out = {"BLEU": corpus_bleu(hyps, refs), "exact": np.mean([h == r[0] for h, r in zip(hyps, refs)])}
    for b, items in by_len.items():
        out[b] = np.mean([h == t for h, t in items])
    return out

eval_pairs = mt_test[:300]
results = {}
for name, p in [("no attention", p_plain), ("attention", p_attn)]:
    results[name + ", greedy"] = evaluate(lambda s: greedy(p, s)[0], eval_pairs)
    results[name + ", beam k=5"] = evaluate(lambda s: beam_search(p, s), eval_pairs)
print(f"{'model':26s} {'BLEU':>6s} {'exact':>6s} {'short':>7s} {'medium':>7s} {'long':>7s}")
for name, r_ in results.items():
    print(f"{name:26s} {100 * r_['BLEU']:6.1f} {r_['exact']:6.1%} {r_['short (<=10)']:7.1%} {r_['medium (11-15)']:7.1%} {r_['long (16+)']:7.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
buckets = ["short (<=10)", "medium (11-15)", "long (16+)"]
xs = np.arange(3)
for i_, name in enumerate(["no attention, greedy", "no attention, beam k=5", "attention, greedy", "attention, beam k=5"]):
    ax.bar(xs + 0.2 * i_, [results[name][b] for b in buckets], width=0.2, label=name)
ax.set_xticks(xs + 0.3); ax.set_xticklabels(buckets); ax.set_ylabel("exact-match accuracy")
ax.set_title("The bottleneck: no-attention seq2seq degrades with length; attention doesn't"); ax.legend(fontsize=8)
plt.show()

**This is the bottleneck problem, measured.** Without attention, accuracy falls steeply with sentence length: the decoder has to reconstruct every word and its position from a single vector. With attention, length barely matters, because each output word can look directly at the source word it needs.

Beam search helps the weaker model more (it can recover from some early mistakes), but it can't fix information that never made it through the bottleneck. **Better search doesn't compensate for a worse model.**

### 3.6 Attention provides interpretability: soft alignment for free
By inspecting the attention distribution, we see what the decoder was focusing on. **We get (soft) alignment for free!** This is cool because we never explicitly trained an alignment system; the network just learned alignment by itself.

In [ ]:
src = mt_test[[len(s) for s, _ in mt_test].index(max(len(s) for s, _ in mt_test[:300]))][0]
out, atts = greedy(p_attn, src)
A = np.array([a[0] for a in atts])          # each step's attention has shape (1, N): one hypothesis
fig, ax = plt.subplots(figsize=(0.45 * len(src) + 2, 0.45 * len(out) + 1.5))
ax.imshow(A, cmap="Greys")
ax.set_xticks(range(len(src))); ax.set_xticklabels(src, rotation=60, ha="right")
ax.set_yticks(range(len(out))); ax.set_yticklabels(out)
ax.set_xlabel("source (English-like)"); ax.set_ylabel("generated target (Toy)")
ax.set_title("Attention distributions (each row sums to 1)")
plt.tight_layout(); plt.show()
print("source:", " ".join(src)); print("output:", " ".join(out))
inv = {v: k for k, v in LEX.items()}
print("\nEach output word, and the source word it attends to most:")
for i, w in enumerate(out):
    j = int(A[i].argmax())
    print(f"  {w:7s} -> {src[j]:8s} (position {j:2d}, weight {A[i, j]:.2f})" + ("" if inv.get(w) == src[j] else "   (not its dictionary translation)"))

The model translates this sentence perfectly. But read the table: **the attention is usually not on the source word you'd align by hand.** For nouns and adjectives, the most-attended position is typically **one word after** the aligned word. The verb's attention isn't on the source verb at all. Is this a fluke of one sentence? Let's measure over many sentences: for each output content word with a unique dictionary source word, compute (attended position − aligned position).

In [ ]:
inv = {v: k for k, v in LEX.items()}
verb_words = {LEX[v] for v in VERBS_MT}
offsets, verb_offsets = collections.Counter(), collections.Counter()
for s_, t_ in mt_test[:200]:
    o_, at_ = greedy(p_attn, s_)
    if o_ != t_:
        continue
    A_ = np.array([a[0] for a in at_])
    for i, w in enumerate(o_):
        if w in ("le", "un", "."):
            continue
        cands = [j for j, sw in enumerate(s_) if LEX[sw] == w]
        if len(cands) == 1:                                   # unambiguous gold alignment
            off = int(np.clip(A_[i].argmax() - cands[0], -3, 3))
            (verb_offsets if w in verb_words else offsets)[off] += 1
for name, c in [("nouns/adjectives/prepositions", offsets), ("verbs", verb_offsets)]:
    tot = sum(c.values())
    print(f"{name:30s}: " + "  ".join(f"offset {k:+d}: {v / tot:.0%}" for k, v in sorted(c.items())) + f"   (n={tot}; ±3 means 3 or more)")

**Attention weights are over encoder *states*, not words.** Our encoder is a left-to-right LSTM, so the state at position $j$ summarizes words $1..j$. The state just *after* a word still contains it, and it also knows the word's right neighbor. So the model often reads a word from the next position, which is the off-by-one pattern above. And the verb doesn't need attention at all: it's at the end of the target, and the decoder can carry it in its own recurrent state from the encoder's final state. Koehn & Knowles (2017, *Six Challenges for Neural Machine Translation*) report the same off-by-one shift between attention and word alignment in real NMT systems.

So the slide's point, **"we get (soft) alignment for free"**, is true in spirit: the network learns *where to look* with no alignment supervision, and the attention pattern is clearly structured and reorders correctly. But treat attention maps as a **hint, not an explanation**. With bidirectional encoders (which mix both left and right context into every state) and multi-layer models, the relation between attention and alignment gets even looser. See Jain & Wallace 2019, *Attention is not Explanation*, and the reply by Wiegreffe & Pinter 2019. If you need real word alignments, extract them with a dedicated method (e.g., `fast_align`, or alignment heads trained with supervision).

### 3.7 Attention is great!
* Attention **significantly improves NMT performance**: it's very useful to allow the decoder to focus on certain parts of the source.
* Attention provides a **more "human-like" model** of the MT process: you can look back at the source sentence while translating, rather than needing to remember it all.
* Attention **solves the bottleneck problem**: it allows the decoder to look directly at the source.
* Attention **helps with the vanishing gradient problem**: it provides a shortcut to faraway states.
* Attention **provides some interpretability**: by inspecting the attention distribution, we see what the decoder was focusing on. We get (soft) alignment for free, with the caveats measured in §3.6.

---
## 4. Attention variants, and attention as a general technique

We have some **values** $h_1, \dots, h_N\in\mathbb{R}^{d_1}$ and a **query** $s\in\mathbb{R}^{d_2}$. Attention always involves:
1. Computing the **attention scores** $e\in\mathbb{R}^N$ *(there are multiple ways to do this)*
2. Taking softmax to get the **attention distribution** $\alpha = \text{softmax}(e)\in\mathbb{R}^N$
3. Using the attention distribution to take a **weighted sum of values**: $a = \sum_{i=1}^N\alpha_i h_i\in\mathbb{R}^{d_1}$, the **attention output** (sometimes called the **context vector**)

### 4.1 Ways to compute the scores $e\in\mathbb{R}^N$ from $h_1,\dots,h_N$ and $s$
| Variant | Score $e_i$ | Notes |
|---|---|---|
| **Basic dot-product** | $s^\top h_i$ | assumes $d_1 = d_2$; the version we saw first |
| **Multiplicative** (Luong, Pham & Manning 2015) | $s^\top W h_i$, $\ W\in\mathbb{R}^{d_2\times d_1}$ | perhaps better called **bilinear** attention; what we trained above |
| **Reduced-rank multiplicative** | $s^\top(U^\top V)h_i = (Us)^\top(Vh_i)$, $\ U\in\mathbb{R}^{k\times d_2}, V\in\mathbb{R}^{k\times d_1}$, $k\ll d_1,d_2$ | **Remember this when we look at Transformers!** (queries and keys are low-dimensional projections) |
| **Additive** (Bahdanau, Cho & Bengio 2014) | $v^\top\tanh(W_1 h_i + W_2 s)$, $\ W_1\in\mathbb{R}^{d_3\times d_1}, W_2\in\mathbb{R}^{d_3\times d_2}, v\in\mathbb{R}^{d_3}$ | $d_3$ (attention dimensionality) is a hyperparameter; "additive" is a weird name: it's really a feed-forward layer |

(Assignment 3 asks you to think about the relative advantages and disadvantages of these.)

In [ ]:
def scores_dot(s, Hv):                    return Hv @ s
def scores_multiplicative(s, Hv, W):      return Hv @ (W.T @ s)                   # s^T W h_i
def scores_reduced_rank(s, Hv, U, Vm):    return (Hv @ Vm.T) @ (U @ s)            # (U s)^T (V h_i)
def scores_additive(s, Hv, W1, W2, v):    return np.tanh(Hv @ W1.T + W2 @ s) @ v

def attend(scores, Hv):
    alpha = softmax(scores)
    return alpha, alpha @ Hv

d1 = d2 = 64; k_rank, d3, N = 8, 32, 12
r = np.random.default_rng(0)
Hv, s = r.normal(size=(N, d1)), r.normal(size=d2)
W, U_, Vm = r.normal(size=(d2, d1)) / 8, r.normal(size=(k_rank, d2)) / 8, r.normal(size=(k_rank, d1)) / 8
W1, W2, v = r.normal(size=(d3, d1)) / 8, r.normal(size=(d3, d2)) / 8, r.normal(size=d3)
variants = {
    "dot-product": (scores_dot(s, Hv), 0),
    "multiplicative": (scores_multiplicative(s, Hv, W), W.size),
    "reduced-rank (k=8)": (scores_reduced_rank(s, Hv, U_, Vm), U_.size + Vm.size),
    "additive (d3=32)": (scores_additive(s, Hv, W1, W2, v), W1.size + W2.size + v.size),
}
for name, (e_, n_params) in variants.items():
    alpha, a = attend(e_, Hv)
    print(f"{name:20s} params {n_params:5d}   alpha sums to {alpha.sum():.3f}   output shape {a.shape}")
# reduced-rank is exactly multiplicative attention with the low-rank matrix W = U^T V:
print("reduced-rank == multiplicative with W = U^T V:", np.allclose(scores_reduced_rank(s, Hv, U_, Vm), scores_multiplicative(s, Hv, U_.T @ Vm)))

### 4.2 Attention is a general deep-learning technique
We've seen attention improve seq2seq for MT. But **you can use attention in many architectures (not just seq2seq) and many tasks (not just MT)**.

**More general definition:**
> Given a set of vector **values**, and a vector **query**, attention is a technique to compute a **weighted sum of the values, dependent on the query**.

We sometimes say that the **query attends to the values**. In seq2seq + attention, each decoder hidden state (query) attends to all the encoder hidden states (values).

**Intuition:**
* The weighted sum is a **selective summary** of the information contained in the values, where the query determines which values to focus on.
* Attention is a way to obtain a **fixed-size representation of an arbitrary set of representations** (the values), dependent on some other representation (the query).

**Upshot:** attention has become the powerful, flexible, general way of doing **pointer and memory manipulation** in all deep-learning models. In the Transformer (L8), every word's representation is a query that attends to all the other words (**self-attention**), and keys and values are separated (the "fuzzy hashtable" view).

---
## 5. Final projects (course logistics, condensed)

The second half of the lecture is about CS224N's final project. The logistics are course-specific; the **research advice** is useful to any ML engineer, so it's summarized here.

**Course work and grading:** 4 assignments (6% + 3×14% = 48%), final project 49% (proposal 8%, milestone 6%, poster 3%, report 32%), participation 3%. Teams of 1–3. Using existing code/resources is fine, but **you're graded on your value-add**, and you must document what you used.

**Default vs. custom project.** The default project (2024): finish a **minBERT** implementation, fine-tune it for sentiment analysis (Rotten Tomatoes / SST), then extend and improve it (contrastive learning, paraphrasing, regularized optimization…). Choose it if you have limited research experience, want guidance and a goal (and a leaderboard). Choose a **custom** project if you have a research idea you're excited about, or want to experience defining a research goal, finding data and tools, and working out how to evaluate it. The project must substantively involve **both human language and neural networks**. Neither option is the easy way to a good grade; there are strong and weak versions of both.

**Project proposal (3–4 pages):** find a key research paper for your topic and write a summary of it and your takeaways; say what you plan to do and how you'll innovate (with a good halfway milestone); describe the models, the **data** (and how you'll get it), and **how you'll evaluate success**; add an **ethical considerations** paragraph. When reviewing a paper, ask: what were the main novel contributions? Is what makes it work general and reusable, or a special case? Are there flaws or neat details? How does it fit with other work? What further questions does it provoke? Most weak project plans are weak in **nuts-and-bolts** ways: no realistic data, no realistic evaluation, no baselines or ablations.

**Milestone:** a progress report when you're more than halfway done: experiments run, preliminary results, plan for the rest. **Write-up:** quality matters a lot. Abstract, introduction, prior work, model, data, experiments, results, analysis & conclusion.

### 5.1 Finding research topics
Two basic starting points, for all of science:
* **[Nails]** Start with a (domain) problem of interest, and try to find good/better ways to address it than are currently known/used.
* **[Hammers]** Start with a technical method/approach of interest, and work out good ways to extend it, improve it, understand it, or find new ways to apply it.

**Project types** (not exhaustive): (1) apply models to a task of interest, often an existing shared task; (2) implement a complex neural architecture and explore its performance; (3) explore prompting / in-context learning / LM programs; *(these three are expected to have experiments with numbers and ablations)*; (4) **analysis/interpretability**: how a model represents knowledge, which phenomena it handles, what errors it makes; (5) rare theoretical or linguistic projects.

**Where to look:** the ACL Anthology, NeurIPS/ICML/ICLR proceedings (OpenReview), arXiv, past CS224N projects, Papers with Code / NLP-progress leaderboards (not always correct), and best of all, an interesting problem in the world. Herb Simon's advice (via Ed Feigenbaum): *"If you see a research area where many people are working, go somewhere else."* Wayne Gretzky: *"I skate to where the puck is going, not where it has been."*

**Old vs. new deep-learning NLP.** In 2010–2018, most work defined and explored better architectures (e.g., adding a copy mechanism to a summarizer). In 2019–2024 that approach is much rarer: **most work downloads a big pretrained model (which fixes the architecture), and the action is in fine-tuning, domain adaptation, prompting, evaluation, and systems built around it.** Exciting areas listed in the slides: evaluating and improving models for things other than accuracy; robustness and domain shift; what pretrained models have learned and how; good task performance without much data (in-context learning, transfer); bias, trustworthiness, explainability; data augmentation; low-resource languages; the long tail; LM programs (RAG, LangChain, DSPy); **small, efficient models** (pruning, quantization, parameter-efficient fine-tuning, LoRA); compositional and systematic generalization (COGS, ReCOGS, gSCAN).

**Can I use ChatGPT/Claude/Gemini?** Yes, and there are interesting projects that do (analysis, prompt learning, chain-of-thought, larger LM systems). But be realistic about scale (you can't train a GPT-2-size model, probably can't even load T5-11B), and remember you're **evaluated on what you did**, not on the impressive zero-shot output of someone else's model.

### 5.2 Finding data
* Some people **collect their own data**: valuable, and teaches real-world challenges, but scope it so it doesn't eat your project. Websites that provide implicit annotations (likes, stars, ratings, responses) help.
* Most people use an **existing curated dataset**: a fast start, with prior work and baselines. Sources: the **Linguistic Data Consortium** (treebanks, NER, coreference, newswire, speech, parallel text), 🤗 **Hugging Face Datasets**, Papers with Code datasets, **statmt.org** / WMT shared tasks for MT, **Universal Dependencies** for parsing, Kaggle, GLUE/SuperGLUE tasks, the Stanford Sentiment Treebank, and lists of NLP datasets.

The 2024 recommended starting point for most practical projects is the Hugging Face `transformers` library:

In [ ]:
# [Optional] needs: pip install transformers datasets  (a GPU helps; ~5 min on a Colab GPU)
# Fine-tune BERT for sentiment on Rotten Tomatoes: the "2024 NLP" workflow from the slides, made runnable.
try:
    import numpy as np
    from datasets import load_dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                              DataCollatorWithPadding)

    ds = load_dataset("rotten_tomatoes")
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        return {"accuracy": float((np.argmax(logits, -1) == labels).mean())}

    args = TrainingArguments(output_dir="bert-rt", per_device_train_batch_size=32, num_train_epochs=1,
                             learning_rate=2e-5, logging_steps=50, report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["validation"],
                      data_collator=DataCollatorWithPadding(tok),     # pads each batch dynamically
                      compute_metrics=compute_metrics)
    trainer.train()
    print(trainer.evaluate())          # validation set; touch ds["test"] only once, at the very end (Sec. 6)
except Exception as e:
    print("Skipping Hugging Face demo:", e)

---
## 6. Care with datasets and in model development

* Many publicly available datasets are released with a **train / dev / test** structure. **We're all on the honor system to do test-set runs only when development is complete.** Do development testing on the dev set only!
* Splits like this presuppose a fairly large dataset. If there's no dev set, or you want a separate tune set, create one by splitting the training data (weighing its usefulness against the reduction in training-set size).
* **Cross-validation** is a technique for maximizing data when you don't have much.
* A fixed test set ensures all systems are assessed against the same gold data. This is generally good, but problematic when the test set turns out to have unusual properties that distort progress on the task.

### 6.1 Training models and pots of data
We're always interested in **generalization performance on an independent test set**, not seen or used during training.
* You build (estimate/train) a model on a **training set**.
* Often, you then set further **hyperparameters** on another, independent set of data, the **tuning set**. *The tuning set is the training set for the hyperparameters!*
* You measure progress as you go on a **dev set** (development test set, or validation set). If you do that a lot, you **overfit to the dev set**, so it can be good to have a second dev set, the **dev2** set.
* Only at the end, you evaluate and present final numbers on a **test set**. Use the final test set **extremely few times**, ideally only once.

**The train, tune, dev, and test sets need to be completely distinct.** It's invalid to test on material you trained on: you'll get falsely good performance. If you keep running on the same evaluation set, you begin to overfit to it: effectively you're "training" on it, learning what does and doesn't work on that particular set.

**This is not a hypothetical.** Below, every "model" is a coin flip: it guesses randomly on a binary task, so its true accuracy is exactly 50%. We "develop" by trying 500 models and keeping whichever scores best on a 200-example dev set. Then we check the winner on the dev set, a dev2 set, and the test set.

In [ ]:
r = np.random.default_rng(0)
n_dev, n_models = 200, 500
y_dev, y_dev2, y_test = r.integers(0, 2, n_dev), r.integers(0, 2, n_dev), r.integers(0, 2, 2000)
# each "model" is a fixed random predictor (think: one hyperparameter setting of a useless model)
model_seeds = r.integers(0, 10**9, n_models)
def predict(seed, n, split):
    return np.random.default_rng([seed, split]).integers(0, 2, n)

dev_scores = np.array([(predict(s, n_dev, 0) == y_dev).mean() for s in model_seeds])
best = model_seeds[dev_scores.argmax()]
print(f"best of {n_models} random models on dev : {dev_scores.max():.1%}   <- looks like real progress!")
print(f"the same model on dev2              : {(predict(best, n_dev, 1) == y_dev2).mean():.1%}")
print(f"the same model on test              : {(predict(best, 2000, 2) == y_test).mean():.1%}   <- the truth: chance")

plt.figure(figsize=(6, 3))
plt.hist(dev_scores, bins=30); plt.axvline(dev_scores.max(), color="red")
plt.xlabel("dev accuracy of each random model"); plt.title("Selection on a small dev set inflates scores"); plt.show()

The winner looks several points better than chance on dev, purely from selection. This "winner's curse" is why you need **dev2 / test sets that the selection never touched**, and why leaderboards with many submissions overstate progress.

**Cross-validation** when data is scarce: split the data into $k$ folds; train on $k-1$, evaluate on the held-out fold; rotate; average. Every example is used for evaluation exactly once, and the variance across folds tells you how noisy your estimate is.

In [ ]:
def k_fold_indices(n, k, seed=0):
    perm = np.random.default_rng(seed).permutation(n)
    folds = np.array_split(perm, k)
    for i in range(k):
        yield np.concatenate([folds[j] for j in range(k) if j != i]), folds[i]

# Example: k-fold CV of a nearest-centroid classifier on a small synthetic dataset
r = np.random.default_rng(1)
Xc = np.vstack([r.normal(0, 1, (60, 5)), r.normal(0.7, 1, (60, 5))]); yc = np.repeat([0, 1], 60)
accs = []
for tr, te in k_fold_indices(len(yc), 5):
    centroids = np.stack([Xc[tr][yc[tr] == c].mean(0) for c in (0, 1)])
    pred = np.argmin(((Xc[te][:, None] - centroids[None]) ** 2).sum(-1), 1)
    accs.append((pred == yc[te]).mean())
print("5-fold accuracies:", np.round(accs, 3), f" mean {np.mean(accs):.3f} ± {np.std(accs):.3f}")

---
## 7. Getting your neural network to train

* **Start with a positive attitude!** Neural networks want to learn! If the network isn't learning, you're doing something to prevent it from learning successfully.
* **Realize the grim reality:** there are lots of things that can cause neural nets to not learn at all, or to not learn very well. Finding and fixing them ("debugging and tuning") can often take more time than implementing your model. It's hard to work out what these things are, but experience, experimental care, and rules of thumb help!

### 7.1 Experimental strategy
* **Work incrementally!** Start with a very simple model and get it to work! It's hard to fix a complex but broken model. Add bells and whistles one by one and get the model working with each of them (or abandon them).
* **Initially run on a tiny amount of data.** You'll see bugs much more easily on a tiny dataset, and it trains really quickly. Something like **4–8 examples** is good. Often synthetic data is useful for this. **Make sure you can get 100% on this data** (testing on train). Otherwise your model is definitely either not powerful enough, or it is broken.
* **Then train and run on a large dataset.** It should still score close to 100% on the training data after optimization; otherwise consider a more powerful model. **Overfitting to training data is not something to fear when doing deep learning**: these models usually generalize well because distributed representations share statistical strength, regardless of overfitting.
* **But you now want good generalization:** regularize your model until it doesn't overfit on dev data. L2 regularization can be useful, but **normally generous dropout is the secret to success**.

### 7.2 The tiny-batch test
Train the attention model from scratch on just **8 sentence pairs**. A correct implementation should drive the training loss to ~0 and reproduce all 8 exactly.

In [ ]:
tiny = mt_train[:8]
p_tiny = init_attn_seq2seq(seed=3); opt = Adam(p_tiny, lr=5e-3)
bt = mt_batch(tiny)
for step in range(1, 301):
    loss, g = attn_seq2seq_fb(p_tiny, *bt); clip_grads(g); opt.step(p_tiny, g)
    if step in (1, 50, 100, 200, 300):
        print(f"step {step:3d}: loss {loss:.4f}")
print("exact reproduction of the 8 training pairs:", sum(greedy(p_tiny, s)[0] == t for s, t in tiny), "/ 8")

### 7.3 "Too good to be true" is a bug report: a data-leak example
Here's a classic seq2seq bug: building the decoder inputs from the target **without shifting** it right. Then at position $t$ the decoder *receives* the word it's supposed to *predict*. Training loss collapses almost immediately, which looks fantastic. But at test time there's no gold target to peek at, and the model has learned nothing useful.

In [ ]:
def leaky_batch(pairs):
    S, SM, Tin, Tout, TM = mt_batch(pairs)
    return S, SM, Tout.copy(), Tout, TM            # BUG: decoder input = target (no shift)

p_leak = init_attn_seq2seq(seed=4); opt = Adam(p_leak)
p_ok = init_attn_seq2seq(seed=4); opt_ok = Adam(p_ok)
r = np.random.default_rng(0)
for step in range(1, 151):
    batch_pairs = [mt_train[i] for i in r.integers(0, len(mt_train), 64)]
    l_leak, g = attn_seq2seq_fb(p_leak, *leaky_batch(batch_pairs)); clip_grads(g); opt.step(p_leak, g)
    l_ok, g = attn_seq2seq_fb(p_ok, *mt_batch(batch_pairs)); clip_grads(g); opt_ok.step(p_ok, g)
    if step in (10, 50, 150):
        print(f"step {step:3d}: train loss with the bug {l_leak:.4f}   correct {l_ok:.4f}")

def leaky_greedy(p, src, max_len=40):        # at test time the buggy model can only feed its own predictions
    Henc, h, c = encode(p, src); y, out = T2I["<s>"], []
    for _ in range(max_len):
        logp, h, c, _ = decoder_step(p, Henc, np.array([y]), h, c)
        y = int(logp[0].argmax()); out.append(tgt_vocab[y])
        if out[-1] == ".":
            break
    return out
s, t = mt_test[0]
print("\nbuggy model at test time:", " ".join(leaky_greedy(p_leak, s)))
print("correct model (150 steps):", " ".join(greedy(p_ok, s)[0]))
print("gold                     :", " ".join(t))

**Lesson:** a loss that drops much faster than it should is as suspicious as one that won't drop. Common leaks: unshifted targets, a missing causal mask in a Transformer decoder (L8), test examples duplicated in training, and features computed using the label.

### 7.4 Details matter!
* **Look at your data**, collect summary statistics.
* **Look at your model's outputs, do error analysis.**
* Tuning hyperparameters, learning rates, getting initialization right, etc. is often important to the successes of neural nets.

Error analysis. Our attention model makes (almost) no mistakes on this toy test set, so let's analyze the **no-attention** model instead: *what kind* of mistakes does the bottleneck cause?

In [ ]:
n_attn_err = sum(greedy(p_attn, s)[0] != t for s, t in mt_test[:600])
print(f"attention model: {n_attn_err} errors in 600 test sentences")
errors = [(s, t, greedy(p_plain, s)[0]) for s, t in mt_test[:600]]
errors = [(s, t, o) for s, t, o in errors if o != t]
print(f"no-attention model: {len(errors)} errors in 600 test sentences")
kinds = collections.Counter()
for s, t, o in errors:
    if len(o) != len(t):
        kinds["wrong length (dropped/repeated words)"] += 1
    elif sorted(o) == sorted(t):
        kinds["right words, wrong order"] += 1
    else:
        kinds["wrong word choice"] += 1
print(dict(kinds))
for s, t, o in errors[:3]:
    print("\n src:", " ".join(s)); print(" ref:", " ".join(t)); print(" out:", " ".join(o))

### 7.5 A practical checklist
| Stage | Check |
|---|---|
| Data | Look at examples; check label distribution, lengths, duplicates between splits |
| Code | Gradient-check custom layers; **overfit 4–8 examples to 100%**; check shapes and masks |
| Baselines | Have a trivial baseline (majority class, word substitution, n-gram) to beat |
| Training | Plot train *and* dev loss; clip gradients; try LR ×3 and ÷3; fix seeds and run a few |
| Suspicion | Loss too low too fast → look for leaks; dev ≫ test → overfit to dev |
| Analysis | Error analysis by category; ablations (remove one component at a time) |
| Reporting | Test set used once; report mean ± std over seeds; describe exactly what you reused |

---
## 8. PyTorch version: seq2seq with multiplicative attention

The same model as §3, in idiomatic PyTorch. The attention step is a batched matrix multiply: `scores = s @ (H_enc W_a^T)^T`, masked with `-inf` at padding, softmaxed, and used to average `H_enc`.

In [ ]:
# [PyTorch] LSTM encoder-decoder with Luong (multiplicative) attention.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class AttnSeq2Seq(nn.Module):
    def __init__(self, n_src, n_tgt, d=32, hidden=64):
        super().__init__()
        self.src_emb = nn.Embedding(n_src, d, padding_idx=0)
        self.tgt_emb = nn.Embedding(n_tgt, d, padding_idx=0)
        self.encoder = nn.LSTM(d, hidden, batch_first=True)
        self.decoder = nn.LSTM(d, hidden, batch_first=True)
        self.W_a = nn.Linear(hidden, hidden, bias=False)
        self.W_c = nn.Linear(2 * hidden, hidden, bias=False)
        self.out = nn.Linear(hidden, n_tgt)

    def encode(self, src, lens):
        packed = pack_padded_sequence(self.src_emb(src), lens, batch_first=True, enforce_sorted=False)
        H, state = self.encoder(packed)
        H, _ = pad_packed_sequence(H, batch_first=True, total_length=src.shape[1])
        return H, state                                            # (B, N, h)

    def attend(self, S, H, src_mask):
        scores = S @ self.W_a(H).transpose(1, 2)                   # (B, T, N): s_t^T W_a h_i
        scores = scores.masked_fill(~src_mask[:, None, :], float("-inf"))
        alpha = scores.softmax(-1)
        A = alpha @ H                                              # (B, T, h)
        return torch.tanh(self.W_c(torch.cat([A, S], -1))), alpha

    def forward(self, src, lens, tgt_in):
        H, state = self.encode(src, lens)
        S, _ = self.decoder(self.tgt_emb(tgt_in), state)
        O, _ = self.attend(S, H, src != 0)
        return self.out(O)

    @torch.no_grad()
    def greedy(self, src_ids, max_len=40):
        src = torch.tensor([src_ids]); H, state = self.encode(src, torch.tensor([len(src_ids)]))
        y, out = torch.tensor([[T2I["<s>"]]]), []
        for _ in range(max_len):
            S, state = self.decoder(self.tgt_emb(y), state)
            O, _ = self.attend(S, H, src != 0)
            y = self.out(O[:, -1]).argmax(-1, keepdim=True)
            out.append(tgt_vocab[y.item()])
            if out[-1] == ".":
                break
        return out

torch.manual_seed(0)
model = AttnSeq2Seq(len(src_vocab), len(tgt_vocab))
opt_t = torch.optim.Adam(model.parameters(), lr=2e-3)
for epoch in range(8):
    perm = np.random.default_rng(epoch).permutation(len(mt_train))
    model.train()
    for k in range(0, len(perm), 64):
        S, SM, Tin, Tout, TM = mt_batch([mt_train[i] for i in perm[k:k + 64]])
        logits = model(torch.tensor(S), torch.tensor(SM.sum(1).astype(int)), torch.tensor(Tin))
        loss = F.cross_entropy(logits.reshape(-1, len(tgt_vocab)), torch.tensor(Tout).reshape(-1), ignore_index=0)
        opt_t.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt_t.step()
    model.eval()
    hyps = [model.greedy([S2I[w] for w in s]) for s, _ in mt_test[:300]]
    print(f"epoch {epoch + 1}: loss {loss.item():.4f}  BLEU {100 * corpus_bleu(hyps, [[t] for _, t in mt_test[:300]]):.1f}  "
          f"exact {np.mean([h == t for h, (_, t) in zip(hyps, mt_test[:300])]):.1%}")

---
## 9. Summary

| Topic | Takeaway |
|---|---|
| BLEU | Clipped n-gram precisions (n = 1–4), geometric mean, brevity penalty; corpus-level; useful but imperfect (paraphrases score low) |
| Decoding | Greedy can't undo mistakes; **beam search** keeps $k$ hypotheses; normalize by length |
| Bottleneck | One vector for the whole source → long sentences fail (measured: steep drop with length) |
| Attention | Scores → softmax → weighted sum of encoder states, at every decoder step; concatenate with the decoder state |
| Benefits | Better performance, solves the bottleneck, shortcut for gradients, (soft, approximate) alignment |
| Variants | Dot, multiplicative/bilinear, reduced-rank (→ Transformers), additive (an MLP) |
| General definition | A query computes a weighted sum of values; a fixed-size summary of an arbitrary set |
| Data hygiene | Train / tune / dev / dev2 / test distinct; test once; selection on dev inflates scores |
| Training strategy | Start simple; overfit 4–8 examples; scale up; regularize (dropout); look at data and errors |

### Exercises
1. Implement **dot-product** attention (no $W_a$) in `attn_seq2seq_fb`. Compare accuracy and training speed.
2. Implement **additive** (Bahdanau) attention, including its backward pass, and gradient-check it.
3. Add **input feeding**: concatenate the previous step's $o_{t-1}$ to the decoder input. (Hint: this makes the decoder loop step-by-step, since $o_{t-1}$ depends on attention at $t-1$.)
4. Plot BLEU vs. beam size $k\in\{1,2,5,10,20\}$ for the no-attention model. Does very large $k$ ever *hurt*? (It does on real MT: the "beam search curse".)
5. Turn off length normalization in `beam_search`. What happens to output lengths?
7. Make the encoder **bidirectional** (as in L6) and redo the alignment-offset measurement of §3.6. Does the off-by-one shift disappear, or does attention become even less word-like?
6. Make the dev-set-overfitting demo worse: use a smaller dev set or more models. How does the inflation scale?

### References
* Papineni et al. 2002, *BLEU*; Post 2018, *A Call for Clarity in Reporting BLEU Scores* (sacreBLEU); Rei et al. 2020 (COMET).
* Bahdanau, Cho & Bengio 2014, *Neural Machine Translation by Jointly Learning to Align and Translate*.
* Luong, Pham & Manning 2015, *Effective Approaches to Attention-based Neural Machine Translation*.
* Britz et al. 2017, *Massive Exploration of Neural Machine Translation Architectures*.
* Ruder 2017, *Deep Learning for NLP Best Practices* (blog).
* Karpathy 2019, *A Recipe for Training Neural Networks* (blog), an excellent companion to §7.
* The CS224N final project guide and past project reports: cs224n.stanford.edu.

**Next:** Lecture 8, Transformers: what if attention is (almost) all you need?